# AIIJC Puzzle — fixed-B DRUNet50/t60 reproduction

This notebook is an intentionally noncanonical CUDA code reference for the single frozen legal arm: bilateral buddies96 strict upright permutation → RGB/luma harmonization → official colour DRUNet sigma50 independently on each 20×20 tile (same-tile reflect pad +4, exact crop) → independent colored NLM h20/h28/h50 → the exact t60 h28-protected/h50-flat blend.

The production authorization remains absent and fail-closed until both frozen all700 calibration and unchanged all700 holdout reports pass `[0.27, 0.28]`, provenance, safety, flatness, and root manual review. The method status is always `METHOD_COMPLIANT_LAYOUT_ACCURACY_UNPROVEN`: a strict bijection does not prove the hidden correct layout.

**Backend disclosure:** canonical submission bytes are produced and independently recomputed on Apple MPS. The strict runtime preflight intentionally blocks Colab Linux/CUDA before test access, so this notebook cannot create an authorized ZIP. Its later CUDA cells remain code reference only; CUDA kernels may differ by 1 LSB and their bytes or hashes must never substitute for canonical MPS artifacts.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/aiijc-puzzle')
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), (
    'Upload or clone the prepared workspace to /content/aiijc-puzzle first.'
)
%cd /content/aiijc-puzzle
%pip install -q -e . torch


## Fail-closed promotion gate — before any test access

The future root authorization must already exist at `configs/compliant_fixed_b_standard_submission_v1.json` and bind exact immutable paths and SHA-256 values for the measurement config, the final production runtime preflight (file SHA plus internal digest), and calibration/holdout commitments, receipts, reports, and manual reviews. The runtime preflight binds the complete transitive production source roster, dependency versions, canonical MPS host identity, and MPS availability. This notebook does not create, guess, or chmod those records. Its canonical loader call below must fail closed on Colab before test access; run the production command on the frozen Apple MPS host instead.


In [ ]:
import json
import os

from aiijc_puzzle.compliant_fixed_b_standard_submission import (
    DEFAULT_PROMOTION_CONFIG,
    PROMOTED_ARM,
    SAFETY_REFERENCE,
    load_promotion_evidence,
)

assert DEFAULT_PROMOTION_CONFIG.is_file(), (
    'BLOCKED: no authorized calibration700 + unchanged holdout700 evidence package'
)
try:
    promotion = load_promotion_evidence()
except ValueError as error:
    assert 'runtime differs from frozen preflight' in str(error) or (
        'requires an available MPS backend' in str(error)
    )
    raise RuntimeError(
        'EXPECTED STOP: Colab CUDA cannot satisfy the canonical MPS preflight'
    ) from error
raise RuntimeError('STOP: authorized production must run on canonical Apple MPS')


## Official KAIR checkpoint

Download only the official KAIR release checkpoint and require the exact SHA-256. KAIR commit: `fc1732f4a4514e42ce15e5b3a1e18c828af47a1e`; license: MIT.


In [ ]:
import hashlib
import urllib.request

CHECKPOINT_URL = 'https://github.com/cszn/KAIR/releases/download/v1.0/drunet_color.pth'
CHECKPOINT_SHA256 = '479abe3c5327dfd10ff54a80ec7d4098ca80752a5c9492cdff31cee430bec4b4'
CHECKPOINT = PROJECT_ROOT / 'artifacts/pretrained-denoisers/kair-fc1732f/drunet_color.pth'
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
checkpoint_valid = CHECKPOINT.is_file() and (
    hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() == CHECKPOINT_SHA256
)
if not checkpoint_valid:
    temporary = CHECKPOINT.with_suffix('.download')
    urllib.request.urlretrieve(CHECKPOINT_URL, temporary)
    observed = hashlib.sha256(temporary.read_bytes()).hexdigest()
    assert observed == CHECKPOINT_SHA256, (observed, CHECKPOINT_SHA256)
    os.replace(temporary, CHECKPOINT)
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() == CHECKPOINT_SHA256
print('Official checkpoint verified:', CHECKPOINT_SHA256)


## Bind the official 700-board test archive

Only after promotion has passed, upload the organizer's original `test.zip` to `/content/test.zip`. Extraction rejects nested paths and duplicate or non-PNG members. No targets, references, templates, source lookup, or cross-board pixels are used.


In [ ]:
import shutil
import zipfile

from aiijc_puzzle.compliant_submission import (
    OFFICIAL_FILENAMES_SHA256,
    OFFICIAL_TEST_ARCHIVE_SHA256,
    build_official_input_snapshot,
)

TEST_ZIP = Path('/content/test.zip')
assert TEST_ZIP.is_file(), 'Upload the original organizer test.zip'
assert hashlib.sha256(TEST_ZIP.read_bytes()).hexdigest() == OFFICIAL_TEST_ARCHIVE_SHA256
TEST_INPUTS = Path('/content/aiijc-fixed-b-official-test-inputs')
assert not TEST_INPUTS.exists(), f'Refusing to overwrite {TEST_INPUTS}'
TEST_INPUTS.mkdir()
with zipfile.ZipFile(TEST_ZIP) as archive:
    infos = archive.infolist()
    names = [info.filename for info in infos]
    assert len(names) == len(set(names)) == 700
    assert all('/' not in name and '\\' not in name and name.endswith('.png') for name in names)
    for info in infos:
        destination = TEST_INPUTS / info.filename
        with archive.open(info) as source, destination.open('xb') as target:
            shutil.copyfileobj(source, target)
snapshot = build_official_input_snapshot(TEST_INPUTS, TEST_ZIP)
assert snapshot.file_count == 700
assert snapshot.filenames_sha256 == OFFICIAL_FILENAMES_SHA256
print('Official test snapshot verified')


## Noncanonical CUDA reproduction

This cell exposes no sigma, NLM, threshold, layout, routing, or model-selection knob. Every dirty board is processed independently; all 576 upright input tiles are used exactly once before restoration. Outputs are deliberately isolated and labelled noncanonical.


In [ ]:
import torch
from PIL import Image

from aiijc_puzzle.compliant_fixed_b_standard_submission import predict_fixed_b_standard
from aiijc_puzzle.compliant_submission import load_rgb_png
from aiijc_puzzle.legacy_upgrade import atomic_write_png, deterministic_submission_zip
from aiijc_puzzle.pretrained_tile_denoiser import load_drunet_color

assert torch.cuda.is_available(), 'Choose a Colab GPU runtime'
device = torch.device('cuda')
model = load_drunet_color(CHECKPOINT, device)
assert sum(parameter.numel() for parameter in model.parameters()) == 32_640_960
NONCANONICAL_ROOT = Path('/content/compliant-fixed-b-standard-cuda-noncanonical-v1')
PREDICTIONS = NONCANONICAL_ROOT / 'predictions'
OUTPUT_ZIP = NONCANONICAL_ROOT / 'submission-cuda-noncanonical.zip'
MANIFEST = NONCANONICAL_ROOT / 'NONCANONICAL-CUDA-MANIFEST.json'
assert not NONCANONICAL_ROOT.exists(), f'Refusing to overwrite {NONCANONICAL_ROOT}'
PREDICTIONS.mkdir(parents=True)
rows = []
for index, name in enumerate(snapshot.filenames, start=1):
    image = load_rgb_png(TEST_INPUTS / name, expected_sha256=snapshot.hashes_by_name[name])
    prediction = predict_fixed_b_standard(image, model, device=device)
    assert prediction.audit['passed']
    assert sorted(prediction.layout.tolist()) == list(range(576))
    png_sha256 = atomic_write_png(PREDICTIONS / name, prediction.restored)
    rows.append({
        'filename': name,
        'layout_sha256': hashlib.sha256(
            prediction.layout.astype('<i4').tobytes()
        ).hexdigest(),
        'output_png_sha256': png_sha256,
    })
    print(f'[{index:03d}/700] {name}')
zip_sha256 = deterministic_submission_zip(PREDICTIONS, list(snapshot.filenames), OUTPUT_ZIP)
manifest = {
    'status': 'NONCANONICAL_CUDA_NUMERICAL_REPRODUCTION',
    'not_authoritative_submission_bytes': True,
    'canonical_backend': 'Apple MPS',
    'reproduction_backend': str(device),
    'backend_rounding_may_differ_by_one_lsb': True,
    'method_status': 'METHOD_COMPLIANT_LAYOUT_ACCURACY_UNPROVEN',
    'promoted_arm': PROMOTED_ARM,
    'safety_reference': SAFETY_REFERENCE,
    'promotion_config_sha256': promotion.config_sha256,
    'checkpoint_url': CHECKPOINT_URL,
    'checkpoint_sha256': CHECKPOINT_SHA256,
    'source_archive_sha256': snapshot.source_archive_sha256,
    'filenames_sha256': snapshot.filenames_sha256,
    'submission_zip_sha256': zip_sha256,
    'file_count': len(rows),
    'rows': rows,
}
MANIFEST.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
print('Noncanonical CUDA ZIP:', OUTPUT_ZIP)
print('SHA-256:', zip_sha256)


In [ ]:
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    assert archive.namelist() == list(snapshot.filenames)
    for name in snapshot.filenames:
        with archive.open(name) as stream, Image.open(stream) as image:
            image.load()
            assert image.format == 'PNG'
            assert image.mode == 'RGB'
            assert image.size == (480, 480)
print('Structural PASS: exactly 700 root-only RGB 480x480 PNG files')
print('Reminder: CUDA bytes remain noncanonical and may differ from MPS by 1 LSB.')
